# Stage 2 — Dataset Understanding

## Characterising the Audio Deepfake Detection Dataset

We analyse dataset size, class balance, audio properties (sample rate, duration, etc.), and justify preprocessing decisions.

**Uses original (raw) data** — run `python download_dataset.py` first. This notebook characterises the dataset *before* preprocessing.

## 1. Dataset Source and Structure

**Source:** Kaggle — `adarshsingh0903/audio-deepfake-detection-dataset`

**Download:** `python download_dataset.py`

**Expected structure:** Audio files organised in folders indicating label (e.g. `real/`, `fake/` or `bonafide/`, `spoof/`).

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..")
sys.path.insert(0, str(PROJECT_ROOT))

from config import RAW_DATA_DIR
from src.utils.paths import get_audio_paths_with_labels

In [2]:
# Scan original (raw) data (run download_dataset.py first)
pairs = get_audio_paths_with_labels(RAW_DATA_DIR)
n_total = len(pairs)
n_real = sum(1 for _, l in pairs if l == 0)
n_fake = sum(1 for _, l in pairs if l == 1)

if n_total == 0:
    print("No audio found. Run: python download_dataset.py")
else:
    print(f"Total files: {n_total}")
    print(f"Real (bonafide): {n_real}")
    print(f"Fake (spoof): {n_fake}")
    print(f"Balance ratio (real/fake): {n_real/n_fake if n_fake else float('inf'):.2f}")

Total files: 4447
Real (bonafide): 2274
Fake (spoof): 2173
Balance ratio (real/fake): 1.05


## 2. Audio Properties (Original Data)

We sample files to inspect native sample rate, duration, and basic statistics.

In [3]:
import numpy as np
import soundfile as sf
import librosa

def inspect_audio(path, sr=None):
    """Inspect raw audio; sr=None preserves native sample rate."""
    info = sf.info(str(path))
    y, _ = librosa.load(str(path), sr=sr, mono=True)
    return {
        "sample_rate": info.samplerate,
        "duration_sec": len(y) / info.samplerate,
        "channels": info.channels,
        "rms": np.sqrt(np.mean(y**2)),
    }

# Sample up to 100 files per class from original data
if n_total > 0:
    real_paths = [p for p, l in pairs if l == 0][:100]
    fake_paths = [p for p, l in pairs if l == 1][:100]
    real_stats = [inspect_audio(p) for p in real_paths]
    fake_stats = [inspect_audio(p) for p in fake_paths]

In [4]:
import pandas as pd

if n_total == 0:
    print("No audio found. Run: python download_dataset.py")
else:
    df_real = pd.DataFrame(real_stats)
    df_fake = pd.DataFrame(fake_stats)
    if len(df_real) > 0 and len(df_fake) > 0:
        print("Real (bonafide) — original audio properties:")
        print(df_real.describe())
        print("\nFake (spoof) — original audio properties:")
        print(df_fake.describe())
    else:
        df_any = df_real if len(df_real) > 0 else df_fake
        label = "Real" if len(df_real) > 0 else "Fake"
        print(f"{label} audio properties (only one class found):")
        print(df_any.describe())

Real (bonafide) — original audio properties:
       sample_rate  duration_sec  channels         rms
count        100.0    100.000000     100.0  100.000000
mean       24000.0      8.190797       1.0    0.064704
std            0.0      4.734712       0.0    0.020913
min        24000.0      3.030042       1.0    0.022699
25%        24000.0      4.117500       1.0    0.049371
50%        24000.0      6.664958       1.0    0.062547
75%        24000.0     11.145000       1.0    0.075342
max        24000.0     25.690000       1.0    0.143909

Fake (spoof) — original audio properties:
        sample_rate  duration_sec  channels         rms
count    100.000000    100.000000     100.0  100.000000
mean   17331.000000      6.848004       1.0    0.095051
std     2518.815859      3.790934       0.0    0.035666
min    16000.000000      2.262500       1.0    0.016385
25%    16000.000000      3.850000       1.0    0.069453
50%    16000.000000      5.743750       1.0    0.089921
75%    16000.000000      

## 3. Preprocessing Justification

| Decision | Choice | Justification |
|----------|--------|---------------|
| **Sample rate** | 16 kHz | Standard for speech; matches Hugging Face models (e.g. Wav2Vec2). Reduces compute vs 44.1 kHz without losing speech-relevant information. |
| **Normalisation** | Peak | Reduces level differences across recordings; avoids model learning from loudness alone. |
| **Trimming** | librosa trim @ 25 dB | Removes leading/trailing silence; focuses on speech content. |
| **Duration** | 5 s clip/pad | Fixed length for batching; 5 s captures typical utterances; center-crop or zero-pad preserves structure. |

## 4. Risks to Assess

- **Class imbalance:** If real/fake ratio is skewed, use stratified splits and macro F1.
- **Bias toward TTS systems:** Dataset may over-represent specific synthesis methods; generalisation to unseen TTS is uncertain.
- **Domain shift:** Studio vs telephony vs noisy environments; Stage 8 tests robustness.

In [5]:
# Next steps: EDA on raw data → preprocess → extract features → EDA on features
# 1. 03_eda_raw_data.ipynb (EDA on raw audio)
# 2. python scripts/run_preprocessing.py
# 3. python scripts/extract_features.py
# 4. 04_eda_acoustic_features.ipynb (EDA on acoustic features)